# 161 — Activation Norm Analysis

How do activation magnitudes behave? Track residual stream norms, compare
attention vs MLP output magnitudes, measure norm concentration, and
attribute norm growth to specific components.

## Why This Matters

Norm activation analyzes the scale and magnitude of internal representations. Representation norms carry meaningful signal — they indicate component importance, reveal outlier dimensions, and constrain what information can be transmitted through the residual stream.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.activation_norm_analysis import (
    residual_norm_profile,
    component_norm_comparison,
    norm_concentration,
    position_norm_variation,
    norm_growth_attribution,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.arange(15)

In [ ]:
result = residual_norm_profile(model, tokens)
print(f"Overall growth: {result['overall_growth']:.2f}x  Mean rate: {result['mean_growth_rate']:.3f}\n")
for p in result['per_layer']:
    bar = '#' * int(p['mean_norm'] * 5)
    print(f"{p['stage']:12s}: mean={p['mean_norm']:.3f}  range=[{p['min_norm']:.3f}, {p['max_norm']:.3f}]  {bar}")

In [ ]:
result = component_norm_comparison(model, tokens)
for p in result['per_layer']:
    print(f"Layer {p['layer']}: attn={p['attn_norm']:.3f} ({p['attn_fraction']:.0%})  "
          f"mlp={p['mlp_norm']:.3f}  resid={p['residual_norm']:.3f}")

In [ ]:
result = norm_concentration(model, tokens)
for p in result['per_layer']:
    bar = '#' * int(p['concentration'] * 30)
    print(f"Layer {p['layer']}: concentration={p['concentration']:.3f}  "
          f"top5_fraction={p['top5_norm_fraction']:.3f}  {bar}")

In [ ]:
result = position_norm_variation(model, tokens)
for p in result['per_layer']:
    print(f"Layer {p['layer']}: std={p['std']:.4f}  cv={p['cv']:.3f}  "
          f"max/min={p['max_min_ratio']:.2f}")

In [ ]:
result = norm_growth_attribution(model, tokens)
for p in result['per_layer']:
    print(f"Layer {p['layer']}: norm_change={p['norm_change']:+.4f}")
    print(f"  attn_self={p['attn_self_contribution']:.4f}  mlp_self={p['mlp_self_contribution']:.4f}")
    print(f"  pre*attn={p['pre_attn_interaction']:+.4f}  pre*mlp={p['pre_mlp_interaction']:+.4f}  "
          f"attn*mlp={p['attn_mlp_interaction']:+.4f}")